Extract AE33 zipfiles and save data to parquet files.
Provide plotting functions for data visualization.

joerg.klausen@meteoswiss.ch

In [ ]:
import os
from datetime import datetime, timedelta
import polars as pl
import matplotlib.pyplot as plt
from processing.ae33 import AE33

ae33 = AE33()
root = "/product_data/data/pay/Kenya/MKN/"

In [ ]:
# process AE33 data files
# years = ["2022", "2023", "2024"]
years = ["2024"]
months = ["{:02d}".format(mm) for mm in range(1, 13, 1)]
for year in years:
    for month in months:
        source = os.path.join(root, "incoming/ae33/data", year, month)
        target = os.path.join("data", "level1", year, month)
        archive = os.path.join(root, "archive/ae33/data", year, month)
        issues = os.path.join(root, "incoming_with_issues/ae33")
        df, errors = ae33.zipfiles_to_parquet(source=source, target=target, archive=archive, issues=issues, plot=True)
    print(errors)
print("done")

In [ ]:
df = pl.read_parquet("data/level1/2024/03/ae33.parquet")
df.schema
dtm = "dtm"

In [ ]:
cols = [dtm, "BC1", "BC2", "BC3", "BC4", "BC5", "BC6", "BC7"]
display(df.select(cols).describe())
start = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")
end = datetime.now().strftime("%Y-%m-%d")
ae33.plot_aethalometer_data(df, start=start, end=end)

# df_flagged = ae33.flag_spurious_data(df)
# cols_flags = [dtm, "flags_BC1", "flags_BC2", "flags_BC3", "flags_BC4", "flags_BC5", "flags_BC6", "flags_BC7"]
# display(df_flagged.select(cols_flags).describe())
# ae33.plot_aethalometer_data(df_flagged)

In [ ]:
# import polars as pl

# def flag_spurious_data(dataframe, value_threshold=0, consecutive_threshold=2):
#     """
#     Flag spurious data in a polars DataFrame consisting of time series data.

#     Parameters:
#     - dataframe: polars DataFrame
#     - value_threshold: threshold for considering values around zero (default: 0)
#     - consecutive_threshold: threshold for consecutive occurrences (default: 2)

#     Returns:
#     - polars DataFrame with an additional 'spurious_flag' column
#     """
#     spurious_flags = []

#     for column in dataframe.columns:
#         # Identify spurious data based on the specified thresholds
#         spurious_mask = (
#             (dataframe[column] <= value_threshold) & 
#             (dataframe[column].shift(-1) > value_threshold) & 
#             (dataframe[column].shift(consecutive_threshold) > value_threshold)
#         ) | (
#             (dataframe[column] <= value_threshold) & 
#             (dataframe[column].shift(1) > value_threshold) & 
#             (dataframe[column].shift(-consecutive_threshold) > value_threshold)
#         )

#         spurious_flags.append(spurious_mask)

#     # Create a new column 'spurious_flag' in the DataFrame
#     dataframe = dataframe.with_column('spurious_flag', pl.col(spurious_flags).any())

#     return dataframe

# # Example usage:
# # Assuming 'your_time_series_data.csv' is your input CSV file
# input_file = 'your_time_series_data.csv'

# # Read the CSV file into a polars DataFrame
# df = pl.read_csv(input_file)

# # Call the function to flag spurious data
# df_with_spurious_flags = flag_spurious_data(df)

# # Display the resulting DataFrame with spurious flags
# print(df_with_spurious_flags)


In [ ]:
# # Concatenate the individual DataFrames into a single one
# combined_data = pl.concat(dataframes)

# # Assuming you have datetime columns, replace 'datetime_column' with the actual column name.
# datetime_column = 'timestamp'

# # Perform aggregation by datetime_column
# agg_data = (
#     combined_data
#     .with_column(combined_data[datetime_column].cast(pl.Date32))
#     .groupby(datetime_column)
#     .agg(pl.sum(combined_data['value_column']))
#     .sort(datetime_column)
# )